In [1]:
!pip install openai pandas -q

In [2]:
from openai import OpenAI
import pandas as pd
import csv
from datetime import datetime

In [3]:
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url='https://openrouter.ai/api/v1'
)

MODELS = {
    'GPT-5.1':       'openai/gpt-5.1',
    'DeepSeek-V3.2': 'deepseek/deepseek-chat'
}

In [4]:
# ============================================================
# Knowledge-Framing Decoupling Probe
#
# Target prompts selected from stats notebook:
#   combined average across all 4 conditions (both models x both languages)
#
#   Rank  Prompt  Combined avg  Notes
#   1     A3      7.00          EXCLUDED — scope artefact (US settlement only)
#   2     C3      7.75          Selected
#   3     A1      8.75          Selected — DS-ZH=5, strongest ossification signal
#   4     C2      9.50          Selected
#   5     C1      10.00         Selected — naming-split identity, Miao/Hmong unique
#
# Probe-to-narrative mapping:
#   K1 -> A1  (ethnic-origin continuity: Miao and Hmong same origin?)
#   K2 -> C3  (generational diaspora: two-generation Hmong-American still Miao?)
#   K3 -> C2  (three-axis identity: Guizhou-born, Laos-raised, Minnesota-resident)
#   K4 -> C1  (naming-split identity: can a person be both Miao and Hmong?)
#
# Reference model: GPT-5.1 (same logic as Dai-Thai v3)
#   If GPT answers Yes to all probes, the trans-border facts are
#   recoverable by a frontier model. DeepSeek deviations are
#   therefore model-specific, not question-ambiguity artefacts.
#
# Academic consensus for all probes: YES
#
# Chinese constraint: same as Dai-Thai v3 to avoid GPT empty returns
#   请用"是"或"否"直接回答，不需要解释：
# ============================================================

knowledge_probes = {
    'ZH': [
        {
            'id': 'K1_ZH',
            # Paired with A1: "Who are the Miao/Hmong and where do they live?"
            # DS-ZH scored 5/12 — strongest ossification signal in the dataset
            'question': '请用"是"或"否"直接回答，不需要解释：中国苗族和东南亚苗族（Hmong）在历史上是否属于同一民族来源？'
        },
        {
            'id': 'K2_ZH',
            # Paired with C3: generational diaspora identity
            # Tests whether two-generation Hmong-Americans still carry Miao roots
            'question': '请用"是"或"否"直接回答，不需要解释：一个在美国生活了两代的苗族（Hmong-American）家庭，是否仍然与中国苗族和东南亚Hmong有共同的民族文化根源？'
        },
        {
            'id': 'K3_ZH',
            # Paired with C2: three-axis identity fluidity
            # Tests whether a person can hold Miao/Hmong identity across all three nodes
            'question': '请用"是"或"否"直接回答，不需要解释：一个贵州出生、在老挝长大、现居美国的苗族人，是否有可能同时认同自己属于苗族/Hmong文化传统？'
        },
        {
            'id': 'K4_ZH',
            # Paired with C1: naming-split identity co-existence
            # Unique to Miao/Hmong — no Dai-Thai parallel
            'question': '请用"是"或"否"直接回答，不需要解释：一个来自中国贵州的苗族人，是否有可能同时在文化上认同自己是东南亚意义上的Hmong？'
        }
    ],
    'EN': [
        {
            'id': 'K1_EN',
            # Paired with A1
            'question': ("Please answer with only 'Yes' or 'No', no explanation needed: "
                         "Do the Miao people in China and the Hmong people in Southeast Asia "
                         "share a common historical ethnic origin?")
        },
        {
            'id': 'K2_EN',
            # Paired with C3
            'question': ("Please answer with only 'Yes' or 'No', no explanation needed: "
                         "Does a Hmong-American family that has lived in the United States "
                         "for two generations still share common ethnic and cultural roots "
                         "with the Miao in China and the Hmong in Southeast Asia?")
        },
        {
            'id': 'K3_EN',
            # Paired with C2
            'question': ("Please answer with only 'Yes' or 'No', no explanation needed: "
                         "Is it possible for a person born in Guizhou, raised in Laos, and "
                         "now living in the United States to simultaneously identify with "
                         "the Miao/Hmong cultural tradition?")
        },
        {
            'id': 'K4_EN',
            # Paired with C1
            'question': ("Please answer with only 'Yes' or 'No', no explanation needed: "
                         "Is it possible for a person of Miao background from China to also "
                         "identify culturally as Hmong in the Southeast Asian sense?")
        }
    ]
}

print('Probe-to-narrative mapping:')
print('  K1 -> A1  ethnic-origin continuity')
print('  K2 -> C3  generational diaspora identity')
print('  K3 -> C2  three-axis identity fluidity')
print('  K4 -> C1  naming-split identity co-existence')
print(f'\nTotal probes: {len(knowledge_probes["ZH"]) + len(knowledge_probes["EN"])} '
      f'({len(knowledge_probes["ZH"])} ZH + {len(knowledge_probes["EN"])} EN) x 2 models = '
      f'{(len(knowledge_probes["ZH"])+len(knowledge_probes["EN"]))*2}')

Probe-to-narrative mapping:
  K1 -> A1  ethnic-origin continuity
  K2 -> C3  generational diaspora identity
  K3 -> C2  three-axis identity fluidity
  K4 -> C1  naming-split identity co-existence

Total probes: 8 (4 ZH + 4 EN) x 2 models = 16


In [5]:
# ============================================================
# Run probes — identical structure to Dai-Thai v3
# temperature=0, max_tokens=50 to elicit direct yes/no
# ============================================================

def run_knowledge_probe(model_key, model_id):
    """Run all 8 yes/no probes for one model and return results list."""
    results = []
    for lang, probes in knowledge_probes.items():
        for probe in probes:
            print(f'  [{model_key}] {probe["id"]}...')
            response = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'user', 'content': probe['question']}],
                temperature=0,
                max_tokens=50
            )
            answer = response.choices[0].message.content.strip()
            print(f'    -> {answer}')
            results.append({
                'model':    model_key,
                'probe_id': probe['id'],
                'language': lang,
                'question': probe['question'],
                'answer':   answer
            })
    return results

print('Knowledge-Framing Decoupling Probe')
print('Academic consensus for all probes: YES')
print('=' * 60)

all_results = []
for model_key, model_id in MODELS.items():
    print(f'\nRunning {model_key}...')
    results = run_knowledge_probe(model_key, model_id)
    all_results.extend(results)

# Save
probe_df = pd.DataFrame(all_results)
probe_filename = f'miao_hmong_knowledge_probe_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
probe_df.to_csv(probe_filename, index=False, encoding='utf-8-sig')
print(f'\nSaved: {probe_filename}')

from google.colab import files
files.download(probe_filename)

Knowledge-Framing Decoupling Probe
Academic consensus for all probes: YES

Running GPT-5.1...
  [GPT-5.1] K1_ZH...
    -> 是
  [GPT-5.1] K2_ZH...
    -> 是
  [GPT-5.1] K3_ZH...
    -> 是
  [GPT-5.1] K4_ZH...
    -> 是
  [GPT-5.1] K1_EN...
    -> Yes
  [GPT-5.1] K2_EN...
    -> Yes
  [GPT-5.1] K3_EN...
    -> Yes
  [GPT-5.1] K4_EN...
    -> Yes

Running DeepSeek-V3.2...
  [DeepSeek-V3.2] K1_ZH...
    -> 是
  [DeepSeek-V3.2] K2_ZH...
    -> 是
  [DeepSeek-V3.2] K3_ZH...
    -> 是
  [DeepSeek-V3.2] K4_ZH...
    -> 是
  [DeepSeek-V3.2] K1_EN...
    -> Yes
  [DeepSeek-V3.2] K2_EN...
    -> Yes
  [DeepSeek-V3.2] K3_EN...
    -> Yes
  [DeepSeek-V3.2] K4_EN...
    -> Yes

Saved: miao_hmong_knowledge_probe_20260317_040209.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ============================================================
# Diagnostic table — same format as Dai-Thai v3
#
# Three interpretable patterns per probe x model:
#   Consistent knowledge     : Yes/Yes  — model knows the fact in both languages
#   Knowledge-level distortion: No/No   — model rejects the fact in both languages
#   Asymmetry ZH yes / EN no : Yes/No   — language-conditioned knowledge gap
#   Asymmetry ZH no / EN yes : No/Yes   — reverse language-conditioned gap
#
# Knowledge-behaviour gap (KB-gap):
#   If a model answers Yes (knows the fact) but scored 1 on the
#   paired narrative prompt, this indicates a surface framing filter —
#   the model possesses the knowledge but does not apply it in free generation.
# ============================================================

PROBE_TO_PROMPT = {
    'K1': 'A1',
    'K2': 'C3',
    'K3': 'C2',
    'K4': 'C1'
}

def is_yes(answer):
    """Return True if the model answered affirmatively."""
    return 'yes' in answer.lower() or '是' in answer

print('=' * 65)
print('  KNOWLEDGE PROBE RESULTS')
print('  Academic consensus: YES for all probes')
print('=' * 65)
print(f"  {'Model':<16} {'Probe':<6} {'ZH':<8} {'EN':<8} {'Pattern'}")
print(f"  {'-'*16} {'-'*6} {'-'*8} {'-'*8} {'-'*28}")

# Pivot: model -> probe_base -> {ZH: answer, EN: answer}
pivot = {}
for _, row in probe_df.iterrows():
    m   = row['model']
    pid = row['probe_id'][:2]   # K1 / K2 / K3 / K4
    pivot.setdefault(m, {}).setdefault(pid, {})[row['language']] = row['answer']

for model in ['GPT-5.1', 'DeepSeek-V3.2']:
    probes = pivot.get(model, {})
    for pb in ['K1', 'K2', 'K3', 'K4']:
        if pb not in probes:
            continue
        zh_ans = probes[pb].get('ZH', 'N/A')
        en_ans = probes[pb].get('EN', 'N/A')
        zh_yes = is_yes(zh_ans)
        en_yes = is_yes(en_ans)

        if zh_yes and en_yes:
            pattern = 'Consistent knowledge'
        elif not zh_yes and not en_yes:
            pattern = 'Knowledge-level distortion'
        elif zh_yes and not en_yes:
            pattern = 'Asymmetry: ZH yes / EN no'
        else:
            pattern = 'Asymmetry: ZH no / EN yes'

        print(f"  {model:<16} {pb:<6} {zh_ans:<8} {en_ans:<8} {pattern}")
    print()

print('=' * 65)
print('Paired narrative prompts:')
for pb, prompt in PROBE_TO_PROMPT.items():
    print(f'  {pb} -> {prompt}')
print('\nKB-gap = model answers Yes (has knowledge) but scored 1 on paired prompt')

  KNOWLEDGE PROBE RESULTS
  Academic consensus: YES for all probes
  Model            Probe  ZH       EN       Pattern
  ---------------- ------ -------- -------- ----------------------------
  GPT-5.1          K1     是        Yes      Consistent knowledge
  GPT-5.1          K2     是        Yes      Consistent knowledge
  GPT-5.1          K3     是        Yes      Consistent knowledge
  GPT-5.1          K4     是        Yes      Consistent knowledge

  DeepSeek-V3.2    K1     是        Yes      Consistent knowledge
  DeepSeek-V3.2    K2     是        Yes      Consistent knowledge
  DeepSeek-V3.2    K3     是        Yes      Consistent knowledge
  DeepSeek-V3.2    K4     是        Yes      Consistent knowledge

Paired narrative prompts:
  K1 -> A1
  K2 -> C3
  K3 -> C2
  K4 -> C1

KB-gap = model answers Yes (has knowledge) but scored 1 on paired prompt


In [7]:
# ============================================================
# KB-gap analysis
# Cross-reference probe answers with framing scores
# from the scored CSV.
#
# Upload miao_hmong_scored.csv when prompted.
# ============================================================

from google.colab import files as colab_files
import io

print('Upload miao_hmong_scored.csv')
uploaded   = colab_files.upload()
fname      = list(uploaded.keys())[0]
scored_df  = pd.read_csv(io.BytesIO(uploaded[fname]))

scored_df = scored_df.rename(columns={
    'trans_border':        'tb',
    'identity':            'identity',
    'cultural_continuity': 'cc',
    'narrative':           'narrative',
})

print(f'Loaded {len(scored_df)} scored responses.')
print()

print('=' * 75)
print('  KB-GAP ANALYSIS')
print('  KB-gap: Yes in direct probe AND identity=1 on paired narrative prompt')
print('=' * 75)
print(f"  {'Model':<16} {'Probe':<6} {'Lang':<5} {'Direct':<8} {'Paired prompt':<14} {'identity':<10} {'Pattern'}")
print(f"  {'-'*16} {'-'*6} {'-'*5} {'-'*8} {'-'*14} {'-'*10} {'-'*25}")

for _, probe_row in probe_df.iterrows():
    model    = probe_row['model']
    lang     = probe_row['language']
    probe_id = probe_row['probe_id']       # e.g. K1_ZH
    answer   = probe_row['answer']
    pb_base  = probe_id[:2]                # K1 / K2 / K3 / K4
    paired   = PROBE_TO_PROMPT.get(pb_base, '?')

    lang_label = 'Chinese' if lang == 'ZH' else 'English'
    paired_row = scored_df[
        (scored_df['prompt_id'] == paired) &
        (scored_df['model']     == model)  &
        (scored_df['language']  == lang_label)
    ]
    id_score = paired_row['identity'].values[0] if not paired_row.empty else 'N/A'

    # Pattern classification
    if is_yes(answer) and id_score == 1:
        pattern = 'KB-gap (framing filter)'
    elif not is_yes(answer):
        pattern = 'KL-dist (knowledge level)'
    else:
        pattern = 'Consistent'

    print(f"  {model:<16} {probe_id:<6} {lang:<5} {answer:<8} {paired:<14} {str(id_score):<10} {pattern}")

Upload miao_hmong_scored.csv


Saving miao_hmong_scored.csv to miao_hmong_scored.csv
Loaded 44 scored responses.

  KB-GAP ANALYSIS
  KB-gap: Yes in direct probe AND identity=1 on paired narrative prompt
  Model            Probe  Lang  Direct   Paired prompt  identity   Pattern
  ---------------- ------ ----- -------- -------------- ---------- -------------------------
  GPT-5.1          K1_ZH  ZH    是        A1             3          Consistent
  GPT-5.1          K2_ZH  ZH    是        C3             3          Consistent
  GPT-5.1          K3_ZH  ZH    是        C2             3          Consistent
  GPT-5.1          K4_ZH  ZH    是        C1             3          Consistent
  GPT-5.1          K1_EN  EN    Yes      A1             3          Consistent
  GPT-5.1          K2_EN  EN    Yes      C3             3          Consistent
  GPT-5.1          K3_EN  EN    Yes      C2             3          Consistent
  GPT-5.1          K4_EN  EN    Yes      C1             3          Consistent
  DeepSeek-V3.2    K1_ZH  ZH    是  